In [39]:
import spacy
import re
import time
import warnings

import pandas as pd

from spacy.symbols import ORTH
from tqdm import tqdm
from pathlib import Path
from spacy.language import Language
from heuristic_tokenize import sent_tokenize_rules 

warnings.filterwarnings('ignore')

In [40]:
# Read MIMIC file and specify output directory
data_path = Path("../Data/discharge.csv.gz")

OUTPUT_DIR = Path("../Data/processed/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(data_path, nrows = 200)

# Data cleaning steps
1. Split each note into sections using clinical-note rules
2. Use SciSpaCy to process each section
3. Add custom rules to detect sentence boundaries
4. Treat MIMIC de-identification placeholders as one token
5. Replace newlines inside sentences with spaces
6. Remove empty sentence strings
7. Write each sentence on its own line
8. Separate different notes with a blank line

In [41]:
#setting sentence boundaries
@Language.component("sbd_component")
def sbd_component(doc):
    for i, token in enumerate(doc[:-2]):
        # define sentence start if period + titlecase token
        if token.text == '.' and doc[i+1].is_title:
            doc[i+1].sent_start = True
        if token.text == '-' and doc[i+1].text != '-':
            doc[i+1].sent_start = True
    return doc

In [42]:
#convert de-identification text into one token
def fix_deid_tokens(text, processed_text):
    deid_regex  = r"\[\*\*.{0,15}.*?\*\*\]" 
    if text:
        indexes = [m.span() for m in re.finditer(deid_regex,text,flags=re.IGNORECASE)]
    else:
        indexes = []
    for start,end in indexes:
        processed_text.merge(start_idx=start,end_idx=end)
    return processed_text
    

In [43]:
def process_section(section, note, processed_sections):
    # perform spacy processing on section
    processed_section = nlp(section['sections'])
    processed_section = fix_deid_tokens(section['sections'], processed_section)
    processed_sections.append(processed_section)

In [44]:
def process_note_helper(note):
    # Replace MIMIC-IV-style de-identification placeholder
    note = re.sub(r'_{3,}', ' <PHI> ', note)
    
    note = re.sub(r'(\d+)-\s*\n\s*(\d+)', r'\1-\2', note)
    
    note_sections = sent_tokenize_rules(note)

    processed_sections = []
    section_frame = pd.DataFrame({"sections": note_sections})
    section_frame.apply(process_section, args=(note, processed_sections,), axis=1)

    return processed_sections

In [45]:
def is_structured_block(text):
    """
    Detects sections that should keep their original line breaks,
    such as Past Medical History lists, labs, vitals, and medication lists.
    """
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if len(lines) < 2:
        return False

    all_caps_lines = 0
    lab_lines = 0
    bullet_or_numbered_lines = 0
    colon_lines = 0

    for line in lines:
        clean_line = re.sub(r"\s+", " ", line).strip()

        # Example: ASTHMA/COPD, HYPERTENSION, ATRIAL FIBRILLATION
        if clean_line.isupper() and len(clean_line.split()) <= 8:
            all_caps_lines += 1

        # Example: WBC-7.2 RBC-4.06 Hgb-9.4
        if re.search(r"\b(WBC|RBC|Hgb|Hct|MCV|MCH|MCHC|RDW|Plt|Na|K|Cl|HCO3|Creat|Glucose|Calcium|Phos|Mg|PTT|INR)\b", clean_line):
            lab_lines += 1

        # Example: 1. Medication, - Hypertension
        if re.match(r"^(\d+\.|-|\*)", clean_line):
            bullet_or_numbered_lines += 1

        # Example: GENERAL:, HEENT:, CARDIAC:
        if re.match(r"^[A-Za-z /()]+:", clean_line):
            colon_lines += 1

    score = all_caps_lines + lab_lines + bullet_or_numbered_lines + colon_lines

    return score >= 2

In [46]:
def process_text(sent, note):
    sent_text = sent["sents"].text

    if not isinstance(sent_text, str):
        return

    if len(sent_text.strip()) == 0:
        return

    # If this looks like a structured clinical block, preserve line breaks
    if "\n" in sent_text and is_structured_block(sent_text):
        lines = []

        for line in sent_text.splitlines():
            line = re.sub(r"\s+", " ", line).strip()

            if len(line) > 0:
                lines.append(line)

        note["text"] += "\n".join(lines) + "\n"

    # Otherwise, treat it as normal prose and join broken lines
    else:
        sent_text = sent_text.replace("\n", " ")
        sent_text = re.sub(r"\s+", " ", sent_text).strip()

        note["text"] += sent_text + "\n"

In [47]:
def get_sentences(processed_section, note):
    # get sentences from spacy processing
    sent_frame = pd.DataFrame({'sents': list(processed_section['sections'].sents)})
    sent_frame.apply(process_text, args=(note,), axis=1)

In [48]:
def process_note(note):
    try:
        note_text = note["text"]

        if pd.isna(note_text):
            note["text"] = ""
            return note

        note_text = str(note_text)
        note["text"] = ""

        processed_sections = process_note_helper(note_text)
        ps = pd.DataFrame({"sections": processed_sections})
        ps.apply(get_sentences, args=(note,), axis=1)

        # Post-processing cleanup
        note["text"] = re.sub(r'(\d+)-\s*\n\s*(\d+)', r'\1-\2', note["text"])
        note["text"] = re.sub(r'\n\s*\.\s*\n', '.\n', note["text"])
        note["text"] = re.sub(r'\s+([.,;:?!])', r'\1', note["text"])
        note["text"] = re.sub(r'\n-\n', '\n- ', note["text"])
        note["text"] = re.sub(r'\s+-\s*\n', '\n- ', note["text"])
        note["text"] = re.sub(r'[ \t]+', ' ', note["text"])
        note["text"] = note["text"].strip()

        return note

    except Exception as e:
        print("Error processing note:", e)
        note["text"] = ""
        return note

In [49]:
start = time.time()
tqdm.pandas()

print('Begin reading notes')

print('Number of notes: %d' % len(data.index))
data['ind'] = list(range(len(data.index)))

nlp = spacy.load('en_core_sci_md', disable=['tagger', 'ner'])
nlp.tokenizer.add_special_case("<PHI>", [{ORTH: "<PHI>"}])
nlp.add_pipe("sbd_component", before='parser')

formatted_notes = data.progress_apply(process_note, axis=1)

output_file = OUTPUT_DIR / "discharge.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for text in formatted_notes["text"]:
        if pd.notna(text) and isinstance(text, str) and len(text.strip()) != 0:
            f.write(text.strip())
            f.write("\n<DOC_SEP>\n")

end = time.time()
print (end-start)
print ("Done formatting notes")

Begin reading notes
Number of notes: 200


100%|██████████| 200/200 [01:41<00:00,  1.97it/s]

106.6897919178009
Done formatting notes


In [50]:
with open(output_file, "r", encoding="utf-8") as f:
    text = f.read()


In [51]:
processed_notes = [note.strip() for note in text.split("<DOC_SEP>") if note.strip()]

idx = 54

print("ORIGINAL NOTE")
print("=" * 80)
print(data.loc[idx, "text"][:3000])

print("\n\nCLEANED NOTE")
print("=" * 80)
print(formatted_notes.loc[idx, "text"][:3000])

ORIGINAL NOTE
 
Name:  ___             Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
IV Dye, Iodine Containing Contrast Media / Oxycodone / 
cilostazol / Varenicline
 
Attending: ___
 
Chief Complaint:
Dyspnea, Atrial Fibrillation
 
Major Surgical or Invasive Procedure:
None

 
History of Present Illness:
___ F with pmhx of COPD (nighttime O2), htn, afib who presents 
with dyspnea, currently being treated for COPD and admitted for 
Afib with RVR.  
 The patient went to the ED on ___ and was diagnosed with a 
COPD flare. She was discharged with a prednisone taper 
(currently on 60mg) and azithromycin. This AM she initially felt 
well, then developed dyspnea at rest, worsening with exertion. 
Her inhalers improved her SOB. She felt that these symptoms were 
consistent with her COPD. She saw her PCP ___ today in 
clinic where she was found to be in Afib w/ RVR, rate around 
110-120. She